In [1]:
'''Exercise 4: NumPy Array Operations'''

import numpy as np

# Task 1: Create a 5x5 matrix where border elements are 1 and interior is 0
# TODO: Create the matrix described above
# Hint: Use np.ones and array slicing

border_matrix = np.ones((5, 5))
border_matrix[1:-1, 1:-1] = 0

print("Task 1 — Border Matrix:")
print(border_matrix)


# Task 2: Normalize a random array
# TODO: Normalize each column to have mean=0 and std=1

np.random.seed(42)
random_data = np.random.randn(100, 3)

col_mean = random_data.mean(axis=0)
col_std = random_data.std(axis=0)


# Task 3: Implement linear regression solution using normal equation
# Given X (features) and y (target), compute theta
# theta = (X^T X)^(-1) X^T y
# TODO: Calculate theta_hat using the normal equation
# TODO: Print the estimated coefficients and compare with true_theta

X = np.random.randn(50, 3)
true_theta = np.array([2.5, -1.2, 3.7])
y = X @ true_theta + np.random.randn(50) * 0.1

theta_hat = np.linalg.inv(X.T @ X) @ X.T @ y

print("\nTask 3 — Linear Regression (Normal Equation):")
print(f"  True theta:      {true_theta}")
print(f"  Estimated theta: {theta_hat.round(4)}")
print(f"  Difference:      {np.abs(true_theta - theta_hat).round(4)}")


Task 1 — Border Matrix:
[[1. 1. 1. 1. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 1. 1. 1. 1.]]

Task 3 — Linear Regression (Normal Equation):
  True theta:      [ 2.5 -1.2  3.7]
  Estimated theta: [ 2.5172 -1.1978  3.724 ]
  Difference:      [0.0172 0.0022 0.024 ]


In [3]:
'''Exercise 5: Panda Data Analysis'''

import pandas as pd
import numpy as np

np.random.seed(42)
n_students = 200

data = {
    'student_id': range(1000, 1000 + n_students),
    'major': np.random.choice(['CS', 'Math', 'Physics', 'Biology'], n_students),
    'year': np.random.choice([1, 2, 3, 4], n_students),
    'exam_score': np.random.normal(75, 10, n_students).clip(0, 100),
    'assignments_completed': np.random.randint(0, 11, n_students),
    'hours_studied': np.random.normal(15, 5, n_students).clip(1, 40)
}

df = pd.DataFrame(data)

# Introduce some NaN values
df.loc[np.random.choice(n_students, 10), 'exam_score'] = np.nan
df.loc[np.random.choice(n_students, 5), 'hours_studied'] = np.nan

# Task 1: Data Cleaning and Exploration
# TODO: Display basic information about the dataset
# TODO: Identify and count missing values
# TODO: Fill missing exam_score with the mean score for the student's major
# TODO: Fill missing hours_studied with the median for the student's year

print("=" * 60)
print("TASK 1 — Data Cleaning & Exploration")
print("=" * 60)

print("\n== DataFrame Info ==")
df.info()

print("\n== Statistical Summary ==")
print(df.describe().round(2))


print("\n== Missing Values ==")
print(df.isnull().sum())

df['exam_score'] = df.groupby('major')['exam_score'].transform(
    lambda x: x.fillna(x.mean())
)

df['hours_studied'] = df.groupby('year')['hours_studied'].transform(
    lambda x: x.fillna(x.median())
)

print("\n── Missing Values After Imputation ──")
print(df.isnull().sum())

# Task 2: Analysis
# TODO: Calculate and display the average exam_score by major
# TODO: Find the major with the highest average exam_score
# TODO: Calculate the correlation between hours_studied and exam_score
# TODO: Create a new column 'performance' with categories:
#       'Excellent' (>90), 'Good' (80-90), 'Average' (70-80), 'Needs Improvement' (<70)

print("\n" + "=" * 60)
print("TASK 2 — Analysis")
print("=" * 60)


avg_by_major = df.groupby('major')['exam_score'].mean().round(2)
print("\n── Average Exam Score by Major ──")
print(avg_by_major)

top_major = avg_by_major.idxmax()
print(f"\n── Top Major: {top_major} ({avg_by_major[top_major]:.2f}) ──")

correlation = df['hours_studied'].corr(df['exam_score'])
print(f"\n── Correlation (hours_studied vs exam_score): {correlation:.4f} ──")

def categorize(score):
    if score > 90:
        return 'Excellent'
    elif score >= 80:
        return 'Good'
    elif score >= 70:
        return 'Average'
    else:
        return 'Needs Improvement'

df['performance'] = df['exam_score'].apply(categorize)

print("\n── Performance Category Distribution ──")
print(df['performance'].value_counts())

# Task 3: Advanced Analysis (10 points)
# TODO: For each major and year combination, calculate:
#       - Number of students
#       - Average exam score
#       - Average hours studied
# TODO: Identify top 5 students based on exam_score (handle ties appropriately)
# TODO: Create a pivot table showing average exam_score by major (rows) and year (columns)

print("\n" + "=" * 60)
print("TASK 3 — Advanced Analysis")
print("=" * 60)

major_year_stats = (
    df.groupby(['major', 'year'])
    .agg(
        num_students=('student_id', 'count'),
        avg_exam_score=('exam_score', 'mean'),
        avg_hours_studied=('hours_studied', 'mean'),
    )
    .round(2)
    .reset_index()
)
print("\n── Stats by Major × Year ──")
print(major_year_stats.to_string(index=False))

df['rank'] = df['exam_score'].rank(method='min', ascending=False)
top5 = (
    df[df['rank'] <= 5]
    .sort_values('rank')[['rank', 'student_id', 'major', 'year', 'exam_score']]
    .reset_index(drop=True)
)
print("\n── Top 5 Students (ties included) ──")
print(top5.to_string(index=False))

pivot = df.pivot_table(
    values='exam_score',
    index='major',
    columns='year',
    aggfunc='mean',
).round(2)
pivot.columns = [f'Year {c}' for c in pivot.columns]
print("\n── Pivot Table: Avg Exam Score (Major × Year) ──")
print(pivot)


TASK 1 — Data Cleaning & Exploration

== DataFrame Info ==
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             200 non-null    int64  
 1   major                  200 non-null    object 
 2   year                   200 non-null    int64  
 3   exam_score             190 non-null    float64
 4   assignments_completed  200 non-null    int64  
 5   hours_studied          195 non-null    float64
dtypes: float64(2), int64(3), object(1)
memory usage: 9.5+ KB

== Statistical Summary ==
       student_id    year  exam_score  assignments_completed  hours_studied
count      200.00  200.00      190.00                 200.00         195.00
mean      1099.50    2.62       75.51                   4.70          14.78
std         57.88    1.16        9.58                   3.24           5.25
min       1000.00    1.00   

In [10]:
'''Exercise 6: Data Visualisation'''

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.gridspec as gridspec # Import gridspec

# Define custom palettes and orderings
PALETTE = {
    'CS': '#FF6B6B',      # Reddish
    'Math': '#4ECDC4',    # Teal
    'Physics': '#45B7D1', # Light Blue
    'Biology': '#FFA07A'  # Light Salmon
}

PERF_ORDER = ['Excellent', 'Good', 'Average', 'Needs Improvement']
PERF_PALETTE = {'Excellent': '#8CC63F', 'Good': '#FFD700', 'Average': '#FF8C00', 'Needs Improvement': '#DC143C'}

# Task 1: Distribution Visualization
# TODO: Create a figure with 2 subplots side by side
#       Left: Histogram of exam scores with KDE overlay
#       Right: Box plot of exam scores by major
# TODO: Add appropriate titles, labels, and styling

fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig1.suptitle("Task 1 — Exam Score Distributions", fontsize=15, fontweight='bold', y=1.02)


sns.histplot(data=df, x='exam_score', kde=True, bins=20,
             color='steelblue', edgecolor='white', linewidth=0.6, ax=ax1)
ax1.axvline(df['exam_score'].mean(), color='crimson', linestyle='--', linewidth=1.5,
            label=f"Mean: {df['exam_score'].mean():.1f}")
ax1.set_title("Distribution of Exam Scores")
ax1.set_xlabel("Exam Score")
ax1.set_ylabel("Count")
ax1.legend()

sns.boxplot(data=df, x='major', y='exam_score', hue='major', palette=PALETTE, # Added hue and legend=False
            order=['Biology', 'CS', 'Math', 'Physics'], legend=False,
            linewidth=1.2, flierprops=dict(marker='o', markersize=4, alpha=0.5), ax=ax2)
ax2.set_title("Exam Score Distribution by Major")
ax2.set_xlabel("Major")
ax2.set_ylabel("Exam Score")

plt.tight_layout()
fig1.savefig('task1_distributions.png', dpi=150, bbox_inches='tight')
plt.close(fig1)
print("Saved: task1_distributions.png")


# Task 2: Relationship Visualization
# TODO: Create a scatter plot of hours_studied vs exam_score
# TODO: Color points by major
# TODO: Add a regression line
# TODO: Include appropriate legends, titles, and axis labels

fig2, ax3 = plt.subplots(figsize=(8, 6))
for major, grp in df.groupby('major'):
    ax3.scatter(grp['hours_studied'], grp['exam_score'], # Changed 'ax' to 'ax3'
               color=PALETTE[major], label=major, alpha=0.65, edgecolors='white',
               linewidth=0.4, s=55)

# Overall regression line
m, b = np.polyfit(df['hours_studied'], df['exam_score'], 1)
x_line = np.linspace(df['hours_studied'].min(), df['hours_studied'].max(), 200)
ax3.plot(x_line, m * x_line + b, color='black', linewidth=2, linestyle='--', # Changed 'ax' to 'ax3'
        label=f'Regression  (slope={m:.3f})')

corr = df['hours_studied'].corr(df['exam_score'])
ax3.annotate(f"r = {corr:.3f}", xy=(0.04, 0.93), xycoords='axes fraction',
            fontsize=11, color='dimgray',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')) # Changed 'ax' to 'ax3'

ax3.set_title("Hours Studied vs Exam Score  (coloured by Major)",
             fontsize=13, fontweight='bold')
ax3.set_xlabel("Hours Studied")
ax3.set_ylabel("Exam Score")
ax3.legend(title='Major', framealpha=0.9)

plt.tight_layout()
fig2.savefig('task2_scatter.png', dpi=150, bbox_inches='tight')
plt.close(fig2)
print("Saved: task2_scatter.png")

# Task 3: Advanced Dashboard
# TODO: Create a 2x2 subplot figure containing:
#       1. Bar chart: Average exam score by major
#       2. Count plot: Number of students by year
#       3. Heat map: Correlation matrix of numerical columns
#       4. Violin plot: Exam score distribution by performance category
# TODO: Adjust layout, add titles, and ensure readability

fig3 = plt.figure(figsize=(15, 11))
fig3.suptitle("Student Performance Dashboard", fontsize=16, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 2, figure=fig3, hspace=0.42, wspace=0.35)

ax_bar   = fig3.add_subplot(gs[0, 0])   # 1. Bar: avg score by major
ax_count = fig3.add_subplot(gs[0, 1])   # 2. Count: students by year
ax_heat  = fig3.add_subplot(gs[1, 0])   # 3. Heatmap: correlation matrix
ax_vio   = fig3.add_subplot(gs[1, 1])   # 4. Violin: score by performance

avg_scores = df.groupby('major')['exam_score'].mean().sort_values(ascending=False)
bars = sns.barplot(x=avg_scores.index, y=avg_scores.values,
                   palette=[PALETTE[m] for m in avg_scores.index],
                   edgecolor='white', linewidth=0.8, ax=ax_bar)
for bar, val in zip(bars.patches, avg_scores.values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax_bar.set_title("Avg Exam Score by Major", fontweight='bold')
ax_bar.set_xlabel("Major");  ax_bar.set_ylabel("Average Exam Score")
ax_bar.set_ylim(0, 100)

year_counts = df['year'].value_counts().sort_index()
sns.barplot(x=year_counts.index, y=year_counts.values,
            palette=sns.color_palette("Blues_d", 4),
            edgecolor='white', linewidth=0.8, ax=ax_count)
for bar, val in zip(ax_count.patches, year_counts.values):
    ax_count.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                  str(val), ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax_count.set_title("Number of Students by Year", fontweight='bold')
ax_count.set_xlabel("Year");  ax_count.set_ylabel("Count")

num_cols = ['exam_score', 'assignments_completed', 'hours_studied', 'year']
corr_matrix = df[num_cols].corr()
# Removed mask to ensure all correlations are plotted in a full matrix, which is more common for initial correlation heatmaps.
# mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8},
            ax=ax_heat)
ax_heat.set_title("Correlation Matrix", fontweight='bold')
ax_heat.tick_params(axis='x', rotation=30)
ax_heat.tick_params(axis='y', rotation=0)

sns.violinplot(data=df, x='performance', y='exam_score',
               order=PERF_ORDER, palette=PERF_PALETTE,
               inner='quartile', linewidth=1.1, ax=ax_vio)
ax_vio.set_title("Exam Score by Performance Category", fontweight='bold')
ax_vio.set_xlabel("Performance Category");  ax_vio.set_ylabel("Exam Score")
ax_vio.tick_params(axis='x', rotation=12)

plt.savefig('task3_dashboard.png', dpi=150, bbox_inches='tight')
plt.close(fig3)
print("Saved: task3_dashboard.png")


Saved: task1_distributions.png
Saved: task2_scatter.png


/tmp/ipykernel_57771/430645594.py:104: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  bars = sns.barplot(x=avg_scores.index, y=avg_scores.values,
/tmp/ipykernel_57771/430645594.py:115: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=year_counts.index, y=year_counts.values,
/tmp/ipykernel_57771/430645594.py:137: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x='performance', y='exam_score',


Saved: task3_dashboard.png


In [12]:
'''Exercise 7: Integration Challenge'''

import numpy as np
import pandas as pd

np.random.seed(42)
n_customers = 500


ages = np.random.randint(18, 70, n_customers)
income = np.random.normal(50000, 20000, n_customers).clip(15000, 150000)
purchase_freq = np.random.poisson(5, n_customers)
avg_purchase_value = np.random.normal(100, 30, n_customers).clip(10, 500)

customers = pd.DataFrame({
    'age': ages,
    'income': income,
    'purchase_frequency': purchase_freq,
    'avg_purchase_value': avg_purchase_value
})

max_freq = customers['purchase_frequency'].max()
customers['churn_risk'] = 1 - (customers['purchase_frequency'] / max_freq)
customers['CLV'] = customers['purchase_frequency'] * customers['avg_purchase_value'] * (1 + customers['churn_risk'])

threshold = customers['CLV'].quantile(0.90)
top10 = customers[customers['CLV'] >= threshold]
